In [4]:
# =============================================================================
# PART 1: TOKENIZATION — turn text into pieces a model can count / embed
# =============================================================================
# What is a token?
#   A token = one small piece of text the model treats as a single unit.
#   Examples (depending on the tokenizer):
#     word-ish:   "paris"           → 1 token
#     subword:    "unhappiness"     → "un" + "happiness"  (2+ tokens)
#     punctuation / special: "." , "?", "<pad>" can be tokens too
#   After tokenization, each token usually gets an integer ID from a vocabulary
#   (e.g. "paris" → 4521). Models do math on IDs / embeddings — not on raw letters.
#
# Why tokenize?
#   Computers don't "read sentences." They work with discrete pieces called tokens.
#   Later steps (Word2Vec, RNNs, Transformers) all start from a token list / token IDs.
#
# This cell = the simplest useful tokenizer:
#   lowercase → find word-like chunks with a regex → list of strings
#
# Not the same as modern LLM tokenizers. Those often use SUBWORD methods:
#
# --- BPE (Byte Pair Encoding) ---
# What: Start from characters/bytes. Repeatedly merge the most common pair
#       ("e"+"r" → "er", "t"+"he" → "the", ...) until you have a vocab of merges.
# Advanced vs whitespace: can split rare/unknown words into known pieces
#       e.g. "unhappiness" → "un" + "happiness" (or finer bits) instead of OOV.
# Who uses it: GPT-2/GPT-3/GPT-4 family, RoBERTa, many Hugging Face models;
#              DeepSeek-style LLMs typically use a BPE-like SentencePiece/BBPE stack.
#
# --- WordPiece ---
# What: Also subword. Similar spirit to BPE, but merges are chosen with a
#       likelihood / language-model style score (not only "most frequent pair").
#       Often shows continuation marks like "##ing" (= suffix "ing").
# Advanced vs whitespace: same win — open vocabulary via pieces; good for names/typos.
# Who uses it: BERT, DistilBERT, many older Google NLP models.
#              (MiniLM is BERT-family → WordPiece-style tokenization under the hood.)
#
# --- Quick contrast ---
#   whitespace (this cell): "Isn't" → isn, t     | whole words only | teaching toy
#   BPE / WordPiece:        may keep or split smarter into subwords | production LLMs
#
# Reminder: Word2Vec is NOT a tokenizer — it embeds tokens AFTER tokenization.

import re
from collections import Counter  # handy later for vocab counts (unused in this tiny demo)

def whitespace_tokenize(text):
    """
    Split text into word tokens.
    Regex: r'\\b\\w+\\b'
      \\b  = word boundary (edge between letter/digit and punctuation/space)
      \\w+ = one or more "word chars" (letters, digits, underscore)
    .lower() so "Paris" and "paris" become the same token.
    """
    return re.findall(r'\b\w+\b', text.lower())


# --- Demo ---
sample = "The capital of France is Paris. Isn't it lovely?"
tokens = whitespace_tokenize(sample)
print(tokens)
# Expected something like:
#   ['the', 'capital', 'of', 'france', 'is', 'paris', 'isn', 't', 'it', 'lovely']
#
# Notice:
#   - Periods / ? removed (punctuation is not kept)
#   - "Isn't" → 'isn' + 't'   because ' is NOT part of \\w in this pattern
#     (a smarter tokenizer might keep "isn't" or "is" + "n't")
#   - Order preserved — tokenization is not a bag-of-words yet; it's a sequence


['the', 'capital', 'of', 'france', 'is', 'paris', 'isn', 't', 'it', 'lovely']


In [ ]:
# =============================================================================
# BPE in CODE (one merge step) — definition was in the previous cell
# =============================================================================
# Tiny corpus idea:
#   words with frequencies = "how often this word appears in training text"
#   We store each word as characters separated by spaces: "low" → "l o w"
#   So adjacent SYMBOLS are easy to spot and merge.
#
# This cell runs exactly ONE BPE step:
#   count adjacent pairs → pick the winner → glue that pair everywhere

def get_stats(vocab):
    """Count how often each neighbor pair appears (weighted by word frequency)."""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()          # "l o w" → ['l', 'o', 'w']
        for i in range(len(symbols) - 1):
            # pair ('l','o') seen in a word with freq 5 → add 5, not 1
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs


def merge_vocab(pair, v):
    """Replace every 'a b' with 'ab' inside all word spellings."""
    bigram = ' '.join(pair)       # ('e', 's') → 'e s'  (what we search for)
    replacement = ''.join(pair)   # ('e', 's') → 'es'   (new glued symbol)
    new_vocab = {}
    for word, freq in v.items():
        # string replace on the spaced spelling, e.g. "n e w e s t" → "n e w es t"
        new_vocab[word.replace(bigram, replacement)] = freq
    return new_vocab


# Start from whole words → split into character tokens
# (freq = pretend corpus counts)
raw = {'low': 5, 'lower': 2, 'newest': 6, 'widest': 3}
vocab = {' '.join(list(word)): freq for word, freq in raw.items()}
print("Initial vocab:", vocab)
# e.g. {'l o w': 5, 'l o w e r': 2, 'n e w e s t': 6, 'w i d e s t': 3}

pairs = get_stats(vocab)
print("Top pairs:", pairs.most_common(3))
# With these freqs, ('e','s') and ('s','t') lead (newest+widest), not ('l','o').

best_pair = pairs.most_common(1)[0][0]   # the pair tuple, e.g. ('e', 's')
vocab = merge_vocab(best_pair, vocab)
print(f"After merging {best_pair}:", vocab)
# e.g. 'n e w e s t' → 'n e w es t',  'w i d e s t' → 'w i d es t'
#
# Real BPE training: repeat get_stats → merge many times (thousands),
# save the ordered merge list = the tokenizer.

# Why BPE matters: GPT, BERT, and Whisper all use subword tokenizers (BPE or SentencePiece). 
# This handles unknown words gracefully—new words become combinations of known subwords

Initial vocab: {'l o w': 5, 'l o w e r': 2, 'n e w e s t': 6, 'w i d e s t': 3}
Top pairs: [(('e', 's'), 9), (('s', 't'), 9), (('w', 'e'), 8)]
After merging ('e', 's'): {'l o w': 5, 'l o w e r': 2, 'n e w es t': 6, 'w i d es t': 3}


In [11]:
# =============================================================================
# PART 2: WORD2VEC FROM SCRATCH — prepare text → vocab → IDs
# =============================================================================
# What Word2Vec is trying to learn (picture this):
#
#   Imagine a blank map. Every unique word gets a DOT on that map.
#   At the start the dots are random (scattered).
#   After training, related words sit NEAR each other:
#
#        queen •  • king
#                    • royal
#         castle •        • kingdom
#              • army
#                    beautiful •  • large     (weaker link; different neighborhood)
#
#   "Near" means their number-lists (vectors) point in a similar direction —
#   same idea as MiniLM sentence embeddings, but ONE vector PER WORD.
#
#   Why would king and queen become neighbors?
#   Look at our sentences:
#     "the king rules the kingdom"
#     "the queen rules the kingdom"
#   Both show up in almost the SAME slot: ___ rules the kingdom.
#   Skip-Gram trains: "if you see king, guess nearby words like rules/kingdom"
#                     "if you see queen, guess those same neighbors"
#   Same homework → similar vectors → dots move together on the map.
#
# Before training we need:
#   1) a small corpus (toy sentences) — the "homework examples"
#   2) a vocabulary (unique words)    — one map-dot per word
#   3) maps word ↔ integer ID         — models use IDs, not the spelling "king"

# --- 1) Tiny corpus ---
# Each string = one sentence. Theme = royalty / kingdom so related words co-occur.
corpus = [
    "the king rules the kingdom",
    "the queen rules the kingdom",
    "the king and queen are royal",
    "the castle is large and beautiful",
    "royal family lives in the castle",
    "the kingdom has a strong army",
    "the army protects the kingdom",
    "the king leads the army",
]

# --- 2) Flatten all tokens, then unique sorted vocab ---
# sent.split() = simple whitespace tokenizer (Part 1 style)
words = [word for sent in corpus for word in sent.split()]
# words = ['the','king','rules',...,'army']  (with repeats)

vocab = sorted(set(words))
# set → unique words; sorted → stable order (so IDs don't jump around)

# --- 3) Bidirectional lookups ---
word2idx = {w: i for i, w in enumerate(vocab)}  # "king" → 7 (example)
idx2word = {i: w for w, i in word2idx.items()}  # 7 → "king"
vocab_size = len(vocab)                         # how many word vectors we'll learn

print(f"Vocabulary size: {vocab_size}")
print(list(word2idx.keys()))
# Expect ~20-ish unique words: a, and, army, beautiful, castle, family, ...


Vocabulary size: 21
['a', 'and', 'are', 'army', 'beautiful', 'castle', 'family', 'has', 'in', 'is', 'king', 'kingdom', 'large', 'leads', 'lives', 'protects', 'queen', 'royal', 'rules', 'strong', 'the']


In [15]:
# =============================================================================
# SKIP-GRAM training pairs: center word → nearby context word
# =============================================================================
# Word2Vec Skip-Gram idea (slow motion):
#
#   Pick one word as CENTER (the spotlight).
#   CONTEXT = other words sitting close to it in the sentence.
#   Training question: "I see the center — which nearby word might appear?"
#
#   window_size=2 means:
#     look at most 2 words LEFT of center and 2 words RIGHT of center.
#     (Edges of the sentence have a shorter window — that's fine.)
#
#   Sentence:   the   king   rules   the   kingdom
#   positions:   0      1       2      3       4
#
#   Spotlight on king (pos 1), window=2:
#        [the]  [king]  [rules]  [the]  kingdom
#          ^center-1   CENTER   +1       +2
#     pairs to learn:
#        king → the     (left)
#        king → rules   (right)
#        king → the     (right+2)
#     (king → king is skipped — a word is not context for itself)
#
#   Then move the spotlight to rules, kingdom, ... and collect more pairs.
#   Each (center, context) pair = one training example (X[i], y[i]).
#
#   Name "skip-gram": you SKIP the center when listing neighbors, and learn
#   from those nearby grams (word pieces) around it.

# How modern models get context

#       Era	                                       How context works
# Word2Vec Skip-Gram            Nearby words in a small window → fixed word vector
# RNNs                          Read the sequence left→right, hidden state carries context
# Transformers                  Attention — each token can look at many others in the sentence
# (GPT, Claude, BERT, MiniLM)  

# So: Skip-Gram understands a limited kind of context (local co-occurrence → word similarity).
# LLMs understand richer context with attention over the whole prompt — different mechanism.

import numpy as np

def generate_skipgram_data(corpus, word2idx, window_size=2):
    X, y = [], []  # X = center IDs, y = context IDs
    for sent in corpus:
        # Map words → integer IDs using the vocab from the previous cell
        tokens = [word2idx[w] for w in sent.split()]

        for i, center in enumerate(tokens):
            # Window slice around position i (clamped to sentence edges)
            start = max(0, i - window_size)
            end = min(len(tokens), i + window_size + 1)  # +1 because range end is exclusive

            for j in range(start, end):
                if i != j:           # skip the center paired with itself
                    X.append(center)     # input: center word ID
                    y.append(tokens[j])  # target: a neighbor word ID
    return np.array(X), np.array(y)


X, y = generate_skipgram_data(corpus, word2idx, window_size=2)
print(f"Training pairs: {len(X)}")
print("Example (center -> context):")
for i in range(5):
    print(f"  {idx2word[X[i]]} -> {idx2word[y[i]]}")
# First sentence "the king rules the kingdom" often starts like:
#   the -> king, the -> rules, king -> the, king -> rules, king -> the, ...


Training pairs: 128
Example (center -> context):
  the -> king
  the -> rules
  king -> the
  king -> rules
  king -> the


In [ ]:
# =============================================================================
# Word2Vec model: Skip-Gram + Negative Sampling (from scratch)
# =============================================================================
# Goal of each training step (one pair from X, y):
#   Center word should score HIGH with its real context word,
#   and score LOW with random "negative" words that were NOT neighbors.
#
# Picture:
#   center="king", true context="kingdom"  → push their vectors TOGETHER
#   negatives="beautiful","large",...      → push those AWAY from "king"
# After many pairs, words that share neighbors end up nearby on the map.
#
# Two embedding tables (classic Word2Vec):
#   W_in  = vectors used when a word is CENTER
#   W_out = vectors used when a word is CONTEXT / negative
# (Often people export W_in as "the" word vectors after training.)

import matplotlib.pyplot as plt

class Word2Vec:
    def __init__(self, vocab_size, embedding_dim=10, n_negatives=5):
        self.W_in = np.random.randn(vocab_size, embedding_dim) * 0.01
        self.W_out = np.random.randn(vocab_size, embedding_dim) * 0.01
        self.n_negatives = n_negatives

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -20, 20)))

    def forward(self, center_idx, context_idx, negative_indices):
        v_c = self.W_in[center_idx]
        u_pos = self.W_out[context_idx]
        u_neg = self.W_out[negative_indices]

        pos_score = self.sigmoid(np.dot(u_pos, v_c))            # want → 1
        neg_scores = self.sigmoid(-np.dot(u_neg, v_c))          # want → 1
        # (minus inside: real neighbors should have large +dot; fakes small/negative)

        loss = -np.log(pos_score + 1e-8) - np.sum(np.log(neg_scores + 1e-8))
        return loss, v_c, u_pos, u_neg, pos_score, neg_scores

    def backward(self, center_idx, context_idx, negative_indices,
                 v_c, u_pos, u_neg, pos_score, neg_scores, lr=0.025):
        # Positive pair: move u_pos and v_c TOWARD each other
        #   dL/d(u·v) = pos_score - 1  (≤ 0 when score is weak)
        grad_u_pos = (pos_score - 1) * v_c

        # Negative pairs: move each u_neg AWAY from v_c
        #   dL/d(u·v) = 1 - neg_score  (≥ 0 when fake still looks like a neighbor)
        # BUGFIX: old code had a flipped minus here → embeddings exploded to 1e100+
        #         then became NaN → explore cell showed all similarities as 0.0
        grad_u_neg = (1 - neg_scores).reshape(-1, 1) * v_c

        self.W_out[context_idx] -= lr * grad_u_pos
        self.W_out[negative_indices] -= lr * grad_u_neg

        grad_v_c = (pos_score - 1) * u_pos + np.sum(
            (1 - neg_scores).reshape(-1, 1) * u_neg, axis=0
        )
        self.W_in[center_idx] -= lr * grad_v_c

    def train_step(self, center_idx, context_idx, vocab_size, lr=0.025):
        # Avoid sampling the true context as a "negative" (tiny quality win)
        negative_indices = []
        while len(negative_indices) < self.n_negatives:
            n = np.random.randint(0, vocab_size)
            if n != context_idx and n != center_idx:
                negative_indices.append(n)
        negative_indices = np.array(negative_indices)

        loss, v_c, u_pos, u_neg, pos_score, neg_scores = self.forward(
            center_idx, context_idx, negative_indices
        )
        self.backward(
            center_idx, context_idx, negative_indices,
            v_c, u_pos, u_neg, pos_score, neg_scores, lr=lr,
        )
        return loss


np.random.seed(42)  # reproducible toy run
embedding_dim = 10
model_w2v = Word2Vec(vocab_size, embedding_dim, n_negatives=5)
epochs = 200  # fewer epochs + correct grads is enough on this tiny corpus
losses = []
for epoch in range(epochs):
    # light LR decay helps stability
    lr = 0.025 * (1.0 - epoch / epochs)
    total_loss = 0.0
    order = np.random.permutation(len(X))
    for i in order:
        loss = model_w2v.train_step(X[i], y[i], vocab_size, lr=lr)
        total_loss += loss
    losses.append(total_loss / len(X))
    if epoch % 40 == 0:
        print(f"Epoch {epoch}, avg loss: {losses[-1]:.4f}, "
              f"|W_in| mean: {np.linalg.norm(model_w2v.W_in, axis=1).mean():.3f}")

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Word2Vec Training')
plt.grid(True, alpha=0.3)
plt.show()
# Expect: loss trends down; |W_in| stays modest (not 1e100)


In [ ]:
# =============================================================================
# Explore learned embeddings (the "map of word dots")
# =============================================================================
# After training, each row of W_in is that word's vector (10 numbers here).
# Cosine similarity = how aligned two arrows are (same idea as the correction engine).
#   1.0  → same direction (very similar)
#   0.0  → unrelated
#
# If you previously saw ALL 0.0 scores: training had exploded/NaN embeddings.
# Re-run the Word2Vec training cell ABOVE first, then this one.

vectors = model_w2v.W_in

print("Finite embeddings?", np.isfinite(vectors).all(),
      "|W_in| mean:", float(np.linalg.norm(vectors, axis=1).mean()))


def cosine_sim(v1, v2):
    """Cosine with a safe denominator."""
    if not (np.isfinite(v1).all() and np.isfinite(v2).all()):
        return 0.0
    n1 = float(np.linalg.norm(v1))
    n2 = float(np.linalg.norm(v2))
    denom = n1 * n2
    if (not np.isfinite(denom)) or denom < 1e-12:
        return 0.0
    return float(np.dot(v1, v2) / denom)


def most_similar(word, vectors, word2idx, idx2word, top_n=5):
    idx = word2idx[word]
    vec = vectors[idx]
    sims = np.array([
        -np.inf if i == idx else cosine_sim(vec, vectors[i])
        for i in range(len(vectors))
    ])
    top_indices = np.argsort(sims)[-top_n:][::-1]
    return [(idx2word[i], float(sims[i])) for i in top_indices]


print("Similar to 'king':", most_similar('king', vectors, word2idx, idx2word))
print("Similar to 'castle':", most_similar('castle', vectors, word2idx, idx2word))

# Famous analogy (needs richer data than our toy corpus):
#   king - man + woman ≈ queen

# Connection to your product: Word2Vec showed that meaning is captured in vector directions. 
# Your correction engine already uses sentence embeddings. Later, we'll fine-tune embeddings 
# for your domain (sales/technical terms).

Finite embeddings? True |W_in| mean: inf
Similar to 'king': [('the', 0.0), ('is', 0.0), ('and', 0.0), ('are', 0.0), ('army', 0.0)]
Similar to 'castle': [('the', 0.0), ('strong', 0.0), ('and', 0.0), ('are', 0.0), ('army', 0.0)]


In [ ]:
# =============================================================================
# PART 3: RECURRENT NEURAL NETWORKS (RNNs)
# =============================================================================
# What is an RNN? — full picture (come back to this)
#
# ---------------------------------------------------------------------------
# 1) The problem MLP cannot solve alone
# ---------------------------------------------------------------------------
#   MLP = one shot:
#     take ONE vector → spit ONE answer
#   Example: flattened MNIST image → digit.
#
#   But language is a LINE of pieces:
#     "the" then "king" then "rules" then "the" then "kingdom"
#   Meaning depends on ORDER and on WHAT CAME BEFORE.
#   If you only feed one word at a time into a plain MLP, it has amnesia —
#   it does not remember that "the" already happened.
#
# ---------------------------------------------------------------------------
# 2) RNN's fix: a sticky note called h (hidden state)
# ---------------------------------------------------------------------------
#   Think of a student reading a sentence left → right, holding one sticky note.
#
#   Before any word: sticky note is blank          h = [0, 0, 0, ...]
#   Read word 1 ("the"):     update sticky note    h0 = f(word1, blank)
#   Read word 2 ("king"):    update sticky note    h1 = f(word2, h0)
#   Read word 3 ("rules"):   update sticky note    h2 = f(word3, h1)
#   ...
#   After the last word, the sticky note is a SUMMARY of the whole sentence
#   (compressed into a fixed-size list of numbers).
#
#   That sticky note IS the hidden state h.
#   It is NOT storing the raw words. It is a learned "vibe / summary so far."
#
# ---------------------------------------------------------------------------
# 3) One step in slow motion
# ---------------------------------------------------------------------------
#   Inputs to the cell at time t:
#     x_t     = this step's input (e.g. embedding of the current word)
#     h_prev  = sticky note from the previous step
#   Output:
#     h_t     = new sticky note
#
#   Formula (what the code below runs):
#     h_t = tanh( W_xh @ x_t  +  W_hh @ h_prev  +  b_h )
#            \______________/    \________________/
#              "new word"           "old memory"
#   Then tanh keeps numbers roughly in (-1, 1).
#
# ---------------------------------------------------------------------------
# 4) Unroll the loop (same brain, many moments)
# ---------------------------------------------------------------------------
#        x0      x1      x2      x3      x4     ← different inputs over time
#         |       |       |       |       |
#        h0  →   h1  →   h2  →   h3  →   h4   ← memory handed forward
#
#   CRITICAL: W_xh, W_hh, b_h are the SAME at every step.
#   You are not training 5 different networks for 5 words.
#   You train ONE cell, reuse it (that's "recurrent").
#
# ---------------------------------------------------------------------------
# 5) Side-by-side so the half-picture becomes whole
# ---------------------------------------------------------------------------
#   MLP:
#     all inputs arrive TOGETHER as one vector → one forward pass → done
#     no "time"; no memory between steps
#
#   RNN:
#     inputs arrive ONE AFTER ANOTHER
#     each step: look at (current input + memory) → update memory
#     after T steps you have seen the whole sequence (through h)
#
#   Word2Vec:
#     each word has a FIXED vector; no sentence memory
#
#   Attention (later):
#     can look at MANY past words directly every step
#     RNN instead squeezes the past into one sticky note h (can forget details)
#
# ---------------------------------------------------------------------------
# 6) What this cell is (and is not)
# ---------------------------------------------------------------------------
#   This cell ONLY demonstrates the memory update loop on fake random x_t.
#   It is not yet classifying text or predicting the next word.
#   Later RNN cells: use h for a real task (e.g. predict next token / label).
#
# This cell = ONE RNN step formula, run over a tiny fake sequence.

# Simple RNN Cell from Scratch

def rnn_cell(x_t, h_prev, W_xh, W_hh, b_h):
    """
    One time step:
      h_t = tanh( W_xh @ x_t  +  W_hh @ h_prev  +  b_h )

    W_xh @ x_t     → "what does this input say?"
    W_hh @ h_prev  → "what did we remember from before?"
    add + bias, then tanh → keep values roughly in (-1, 1)
    """
    return np.tanh(W_xh @ x_t + W_hh @ h_prev + b_h)


# --- Toy sizes ---
# input_size=4  → each time step is a 4-number vector (pretend "word embedding")
# hidden_size=3 → memory h is only 3 numbers (tiny on purpose so prints are readable)
np.random.seed(0)
input_size = 4
hidden_size = 3
W_xh = np.random.randn(hidden_size, input_size) * 0.1   # (3, 4)
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1  # (3, 3)
b_h = np.zeros(hidden_size)                             # (3,)

# Fake sequence: 5 time steps, each a random 4-d input
X_seq = np.random.randn(5, input_size)  # shape (time=5, input_size=4)

# Start with empty memory
h = np.zeros(hidden_size)
states = []
for t in range(5):
    # Same cell, same weights — only x_t and h change
    h = rnn_cell(X_seq[t], h, W_xh, W_hh, b_h)
    states.append(h.copy())
    print(f"t={t}, h={np.round(h, 3)}")

# How to read the printout:
#   Each line is the MEMORY after seeing tokens 0..t.
#   Numbers are just the 3 hidden units (not "words" yet).
#   Notice h KEEPS CHANGING every step — it depends on all previous inputs
#   through the chain h0→h1→h2→... (that is the recurrence).
#   Values stay in about (-1, 1) because of tanh.
